# GwenLand glcuda Wave 120 - repaired T4 event profile

Whole-prefill GPU timing with post-workload detailed event drain.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave120-in-process-stability-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "11df9cf1d425a9b421a4be12d5ff059b09ba2aec"
PATCH_SHA256 = "956b23ac69d3990893d7ac7d42069a55c69bc644fa54cfdb6a5ee0decf20d153"
PATCH_GZIP_B64 = """H4sIAHsEpmoC/71c61bkOJL+z1Oo2FnGOThN3m8MNU1X0bW9XRe6oHpmt7qOcdoyePAly7JJGIpz9iH2CfdJNiIk23LaCXSf2eUHZFpSSIoIRXwRCuMFvs+63csgY87BZejmnnPAb51oFXJxsHZueL8/s4PYXqWJy4WwReYsgzDI7qxUsOVvHbET8zXzg5CzKPE46/d6k9FoJ4g9fst6z/yxrOVwyL3h0B053ng5HLh9d86nk+mw74xH7sid92b+cOn5k51ut8sOPH5zEOdhuLO/v/87Vvzdd6zbM3tsv2/2xyP23Xc7+wcHL9hfYRiDcSyIu2ocg79e7mZBErOSBLty0hgaLRomx36I5f5Dk6V5HPPUZK8+vT6G8UHkpHfMTeKM32Ymc2KPOWGYuA4RdcKMp7GTcZZdcUkq5ZkTxNyjrh73eZpyr5tyEXi5E7KVk10Ji71KogjGa+vzQ+dSMCflTOSrVRhwT9Jb3iFt5sKsPGVL7ifQpdgfbCrNDlWHXAB9sQ4y94oFgvFboOKCFl3xlMNmd/ZzwRkwGwgsFjy+hFXaWeoE2WJx/yY8oQcm+zGGJf8Yr/Ls4bAcAvKhTvjhVRL7waXJ5Dc5rOqKK1ss3tBf2XaIUwMDRcZOP354d3puf3r/4/mC7YksZUds9x13RJ4iB2HRHgeGRkEciCxwmbgTGY9IjNEqgy2m3Ae9ubPYCWwO2MyukjXLkmsOIndSZFEI4s/4JbAqcrI0uGVRHmYBckJKDFYJbAMVAAlFPErSO5Od81gkKcgEpCRF7Ae30B46eQy8vORJxLMUZt09LHby86fjj+cn52cLIBj8g8M+xr2y8a/HH999Oj2zT08+2vBR61N2Ofnb6cmr85PXtmLJ+YefTt5r1AajEfHNj2E/IAsj8ASw7HM+HHzpsO5LTUzsfmefwU/zCf4Qc2waDb+sLLFvuGt0zKpH5NzaYARs6gnd+lobcH/FUycD+SxYz+rpTcnKvl6w0eazFXacj7WnKV9xJ7NXPIbjcgcTWPoUllUtfLGAA+OAwIyO7PCws/+g2ADn0nbSyJANUn2BI7oWKqoaq9QTF2QaeHBMF2yZJKF6mqSOG8Ij6AhPiKsfuYDZ/+xPRib7Prn9s3eHhsOD45KmSbpYnOCfly8LBstVWIJn9pKDqoCtuLbpzNu+H9vFoTfK+Tt/OZQjQ56xBCR1VNAIkAlGKetO2TPwsaMlj4ASEntxtEWD2Ldv1L0Uu+WA9QTt50YHR32Wm/6iqwhYrDyNGezNAOMCx+aFUTXiz64cxKJAQKt7tQAzFR3dP7DaovBB8Wnxlwe0P9zNuHf0+f7hy65ZJwm7KpnC7tlu+WWXwchQ0MPClMKzjeENjrS0lyzYbKPNaA87mjKC8UiMTuewVD/88+HakBNycJShHYlOpZYRrNDQdQfO1nNUBxUgAgVwUrD6R6pnfLNY4AOjY4nrYGX0O5q6kH+CrthBW3EMfsnQt5Bc20lq7II3uATlfsyZsncfXp+8ZZ+db8svuzXVJCwgp1ITaI0pv+GpwHbSBi5eGNgfNc0DV+MjB87AZBq7y92OPhAWgEjDTmL3OaNVd0mjRgWdwZHuTSxpY4xZjWHA3stVDj11I7FYgIe8sl3yYobu0vQzITj3FnIdo4FuLR8752qA74D+FmMeihXBSkC7AmDlX7QnYeJ4NknW2KM/NTGA8sLqsR/MCq22ewWb3JMc0C0EHvSQx/KMb7EMv+nEFx4XcA8cl4x5aeBnCwZHHCa4f2gc52L+jeftS3nm0atL0lWSrIDFYgFuq9RMt8Ff90n+SlNgS0MsCVRWeK9mhqvu0PVP1UBtA6XB0Z75QSpaj6dESgXJSx6jmwXMESfSiO7WtiGu8sxL1rFRHQUQnGZCAQp9Jr0zWZbmdfuOPW3sAUB9A5ro3UgxlJPdA50zGXLArCYx1WrLlWkmshTXwQH7sBQ8vSG41U3i8A5cHKAntkpAwrQYhdTnFvuJ8xVLAHzD+YVTBB1h2A0vSekbBMlwNBywEcS7JQROALGBsyEgHhY2hngDpzgEF1RSC7JaRHANI8DVMLQHqAKwhtC5C+JLoj/o9bqCIhItdAD+WOWhq5kzjZHSSBW+AtSljasopVaG4ugM1k2QU57+uoysslXXK2oBbtmw9tj4ln1jpcPa7KU0UEph0CvWqs2ax86NE4TOEq2vvjiIh+IsjBvm4jO5mUGvq3gC+nf/627NQ/+6u7h/MH/dvUoEYMGSPfh4Yc2xBXb6dEO2Ui0TbFHnkGagCR6ebZqqmTYaSjbAgc6clg6VsXUEA6jI/oRhM6BjdvD04Cb6OKyfVFC2S6nkJSl6JDbP6hZZ1OVBQ6U0YifiwKNfd+8fft0F3hUL1BiNQWYpqOVdxgX4NscrnkSOK7ZwmXwmzmXhNFsbtzCl6kAr2NpaLcnK43XqrFCRe52t/XHBj/XsNA1Z4ZhrFnfDbwIe3OKoYvSmCNJ0Y1z8pkdfmkDKw/5al/IPWfKa/wGEBJ3B9hQY7L4iUuJmtYgHHUjEN0XKYmM4YDQNcTu75TDUxa+5k2Y0HH1HEfNu+hZjlYgAaWu+ooNjaL3k1+0AzDOgWh7nETk6ODwb6vwsq/m4LyrIwCi5zd8UYhz+1uPVnykHIc9XxWP9lCkWFkeoYFXxHZaq9261mK028XfbQ7Jg5VJbGtWKW1pKOTebYB9t82yxvY/a36dM7KMDG+Z1yzFXf1qEXAkXjogXuFmX4OAzZFwKTGqFsFcQH0gBKzm2CFcuWFE7QIJ1kbWKqjiJLY+AUwN9eIsEtK6jxx/rsUvDIEoriJGw15KxFql7oADWAQDvWm662aYSzsvByJ8v54476U+G897SW8568MUfDgfLaX+45DNv7s78sWVNZrPR3He9/mzquKOZPxwPuOe6Ez4dw5PRcD7u8VG/5xQJbcw7P7K2eia6pR1zzhNzwvYn5hQzzgxTswT8QMhoBddO6rGVA4DUAJOID8FVZR1rh+1gzlEG+L4fLBaufZMEnkqY0mNxF7sQ+GdJFMDf+2P68D0mqtgHNKEAScE6KUJFAvdNSBkFfI6LGwzGsLL9wUAukK3yJRBPAeoCyMatnHHMC0r/xqPIjvsTAlWcEjoyLcYkUD4oEHp/ITPYjDvuFfiVuOsHCLJ/+OE9KzPaaN8JMWOigAGC5ukfRUXKCdFf33X9HJ2Uk2UA8tEPfXx3tv/zjFhmsQ8rPGaAO7Mg1CH68cH31k4XSbVF21our61Z4+PmzsbHmCHfTNVj5vFQ5qFBohBd0LbSZK1CBbZ03GuN1voqkKn2NYTlGIXAcplzCTQF+ExAA5zD8ZP9gYqwiapXLJzENh+a/T7IbT4yB/OnBOfbwD/YItL6ej3CjM6SL1Q/WNgDKAPlSrlIwhvelopUzh2EYX+d2ZdhXs+J8vgmSJM4AiHZPMYAwKu1V+mPBMxjGngwHIQHsvoz9npZJFLxS+Hh9cnY3l4LDQ2jtSxAZdtYAFa1yRMUxL//eE6i4tGSex4I9PT8byRUxQiGcKcWh4LKQXgY2dPxPl3dlKQyugvo4hnDLFgO4R8IGNMIQDZLEmZcvHmLd0P2+w/2u3fHR/0LlqwgsIQI0kS1qUihNpR7xbCyUA1cKs7dY95q5LA3J+/edZSaoPRBgDifQbcubO8V/NETjCUH8ApGZhV1LHXGQ3+xoLwHJbqkDhScJqImew8xd6cRuR+wtzCMlqcUXuQA3ChAZg6LnRTPQrklGdwXlMvIvqIGBsRiFzjXBTIR0XsXenXxQxHDFyddkztIKAzcOxVo/8tnL3GNqwDkGne+KLddcal1kxU3NB5qrq55LrarM/nBJ9lfw59Kb45o9jIFBY8M0EtErLXeAsEqyQXTT4kFmhE5f09Sk9WfBYDrOxtjo8iRSBeIvAQqU5ONO3jGqnzyjQOMEcZuTWt3O1YgbDDoEoijJRrOZuaM7Y96UzRI8KT9uJHKKPi8U4NeKGetz6G028VKmyxnR1WPNjOxdQuvT344+WiDD7I/npz9+PrT8Vu5HYHJ185GKqVt3qftY9uaNkDl71zeBpVHNLGeI6iJXXMmtbuDjZUcn5/DIj789azOn00NGtmVVz7CB1XnR+VA5EGZRk+RT/nlV30O0rdRf46AZTTqS8CyXd10yGK2Pcb7u9beJcAx66rW5PqTYtHhhEw8N/t0NpahiWlzgV+dKSbzA7y5qrXU5SFBQn/aHyGv+tPhxOzPtnOLAM4VB9OaamityO8RNBMscu4UqoN+QVoHc47nEaDTHGKB7NqhW81ttWiz8uHGngC/1IAG8qol9K3mSLKbRoF/F4uPPHRuCQw0vNaZrHkoQCt4qGzNOfh3wNVX4FSCf8A5Kd0Ww1wERVTCKmlUxE4rn0RX9an6EiOGoDIMQRDRYufAmbLoRBWVMIiQBHMqcgIWD75AVpUc0BXEgaomIYRC9QgEOREaAFJmiV/ktruXYbJ0woqY7iSjPKNNPO0mn7yeJtmYrIb3OvoJ3CKijbyyyAA0GdR305Yr0lU8U8lTGQyErqXOlZwdTAF8eLx7CQZRwTcKI1RRjJcGNxhtsMR185UTu3fsaw5Iz5LYejqcYsg2mE57Zn9cgGvEx8mK2xkuSWDySpVdmOwK+G97QVQ+8FP+1V46AmCBPxyQ+hq/cPfP8OWlyX4BfwcSpUhPFPyiqC6HAHCx+BMGZ0o62EeJBRZQ3MoW9UFN62+DRS0qeGzM6wNLQdr2Jd4S1bNncAx5mr0wXjzi1mrpRUJ/uqMsKDxJQP7+/ePVAuiaFJ90Wsm8eOY6tNvW30ynlp9tWQ9FHLo+YkgRRJibj3KBuMINsViL3zpuFkp11OILURnnioxajmD/81//DZaEbr/lzWppspCMtcpuZUkeGgiyDRI+apGnurLCCJMiC/bm9JO1NRETBstGAkY+U4mX+dTn8+XM8af+fDQcTnh/MBrx5aA3Hy973mw8HU0GM+6OLWvc6y97k8HQG4yd/nTW4/OZO3K93nzo9ydzbz4YL+fz5WS8NfGi5m0kXNRzPLjjPh5b+N0f4qGlE0UpNThT92fyEwhNfpB395gf0SLnjWv9km8/UF3Xx/dv6IKfQha8eUSLHyxDNQ0w9LAIW45YFkS8i725pzk7WR+gQoV8MioiBZzk+ycCJPRRfxRlbWDdAWN9YJtbUuuhW9L0RqUmtsRPxa29dFGwcCx+w3rCjfRO5bXIC13gMbjAC9VlggFfGgm6RXUTfhsIugXAq1rlnCpqhTJKjwTaWHdftaUBhAzI7j7hth4vs6iHaNJ34Erkc+QiT7uUZyWHjWkb4BfmoTCIB0iQX16xzxf1shBw0RdfpOeYz1EB+70a2tJ7N5yjShdiFQvVsFROrUIpdBktU0pF4knTCJJ8O3SpAEtFTd2Cl8KtZFoBnarQ9f8GIdSLntqSERgCFNfsR8SpDcygGjfvsQsutlxc2xh7Gt++FfMtFlIkWFKBAgLgi6oaZIETIvd2i8KS2lVRIa8nd17koLSxReK74SZOqEiHXSC4u2CZquSgKh5KoaDAVCaJgCAYAeoBywR7BtAbDk8deH//4Ux2sdhxcSPPHB/0gdS3UN2qyuXii6wGTjGVGmkeQ14KoqNSM8KhX4G15E5ksiWcXh+0LevyGBaLbqeoxgZqTsauktCTmApiLUwzl8ZZHQ11LNDEbTsnqggDzaEAQwGkHbYKVjykk7CGxTE3TOCkgsfQMqt37AJgj1zpRYMaZXFXSZqpLC72J49JN83YjOrgJtGKKn5VBq5BBrYN9gJ3X5iHVZ5Z9aCRCjPU/TVF3Xgpb6mLzchZGd/ENybKK+6OJfKolo5Qc52mRfAF08KOMUJA941JSkxMnuVRRD7d+7vjos1U1/9BGVdoxMBw+3mIsVkagBQparni7rWUaYA10zFyoKyvYZfOSko4yBrUoM9NkOSCanbcJE9hYo/CILy2RaYWrCzrbLpB7IeVd9GIYaGHFCtqojwBmsPS+bz/CJ9XzUs+xWMsC4lES3OZTS5txTOFpWcv8Of0CqD/qdwpu1dUzGp1D1q+q1PexIwosTGYjsx+73lHRGX3Y7Jbxh6Wu5XhcruFrXGLzN4Re0VV+nQnQEVw3XYrXAJTaTeMPRzeKEHa2v+JhCv+EMXNdBl6SVl0aT1mcPWs12amNIDV7JVJUa0RXX/M5dsQ+YqF9CJDGFwXLySAof0sgsj7Qk0LhjUqdADBQ3prUOcaKXLEWFYGwXtIoXYgurKUDBUXzuFNIBAubgfcMtxvYO7ysYLd/ng09Cf94dx1hr3RsD+bev5sOej3puPldDrpjR1vNhs6I8saLYe9/mjWG078ZX/u9/qz0XA6nwxcPp8O5rzPPac/noy9rbC7mrqBvKsmysvNKC03a1wjvuHxeUCW6V6iLeSiUBwurvmUNRbgrySXARgKBL/0ikoS36gUUuigowliSeniA6asLlS8w8F2udeU5kx8FBi7TGCir3mAtR3LRBo23UvQWyBg9XdUloTOFZxdwELof2MlL6YuKku68r4xTtZg6nGn8t2Ms/PjNyf2++N3J2cL9hlfUzlksy9YpLOz/3ivuewl6wi+Xt/smpuOpse65IrZPgOAHsgqSSy5gJX8g8PTnw9+OviFroTUHeouXfnhmJIaXq0AnWXggDH6et3FRpN9TE5P9CHXN/Y6BYtHw2DIAIb89As4BHAQjFpksEU5EvhdyvtJLshJ8NRiQUC1SZhkDJMQLoUGtPZ/l4dGG4IJDDtfqVEwZKKGyAtibN6HM9wYS3sC2demm6qxVW4SeuAbMNpwsie7YWRjdkeTCAyfwXCZI1UywVqLZR7iK1/6Ar7I6NJwUyptKhhk//zTL9UrO70tnY7fV336h8Tz6RB5Ph2VPG8MevOpGjQ53FE36Ria8TDBCtWEoe3BkljtnMhzI/BsXKwTunQ7LlPJ8IjkdSGpQRsGntAVqIJ3v1SFrxGZuwJQlHfgaoYsSawt2/xQrXh6SCel0eet9i7U7LCI2s4JqnkAPVxOl4SIDNc8uLzKZGIvoNey3DSBSBXxROJTKa/EhJiCxkiZEuXjGSb69vuTIuFXJKUxx4baB9Yj9iAG5cKoX2+r7LhaH3leCOvR1EHcxDHPaLzmy/zSZK9AAOBeXst6AYikdCN5KnPtJWbQCw/CsCsBLlaHEJYwgddhGAgOHMJKgW4Rk0WwkM8qyvUno5d49MwqZGs2z79odQmvdVaiWdZnBKATYeSJ2Wx5LZ7yLpVUVtPTSJgCOLAxc62lNin4BpiNCkVkCWvqxMxQdw63gErz+BpwVjUJpdJbJ6m11CZ5J1/mu+s6oBdRHlIYUW7OYj/Q2wxd9K4rAPOg+Saqd1CGAFphSoFwKMB58/2BKLAyus7iAhwZgwfD52t8+5I4ijmRMEmuK1Lc9wM3wKMILilE61dUkQv0gaBv2VXE6Y3GMAFCuqQdt50JekONB6fy1Qz1JhgFOwqMwyYxuNLSVMU7fTKdrQkrzcEHXPFYySoj4bm4WIgDI/nyK8UjghllkGZq+F1dTI0oUTKdKKj7rJPgpRiEyQSWIg0GXCmjOqB3BRZI5QuvClbAU43rt9yVIY8sjCVBymsagDSILrT1WgBUCHfK9K1w7iR80JhVhpm1ihd5p5aEvFvco1GsFkIYhcmkoGAZpsaqeI54B0gySC32KUYoqqXySiSFaJ1oXphSkPzWDXNPGV0KoFz8lfilItJ7xiUnMY8ENkybTqj0ABJYJ+k1QnaCQngAAERjFnO/Uo8qjiqzaj5lM6uk2uukuKGSNhnANMCTX6RhwWmCOJvpaEag2XevZCZt0EM4OR9MTC2Ttsrf0cts9zXc/QMoQvcylZkx9E0iX3YLl2ZQLjXiTgxcwJC3zKiUb3wkcWdRI1hgCUJTpsQk+NksgSaV2MfZOqDCOjgTBkGxAwD++UGNFgKEA8fzOpS8B/eL74BjgonojPvdyfRfadEkMIl2HSyxFqsQzI8AiCqsekhmYJyX2Ze5yeQnLy4+cXAsWBVSj8keH1B8DKONoWW1AirrYvE6l8nExeI/Tz5+2Lh+/n/sR+oBsTJYjv35eFILltsVBE/1ISvqr85P3p68Ozn/+B9YguWIayrak0gSX7qy6rHjGsRqF3byiJSGUWJgS3FDSf2R4oYNmrWve3uyfEcaUbt8n8bo4LTYayNAp6AfjbHd5vwRd+MNAH7eOlC6J+jZU+5ka095jf2cnuiG6h33n7vqubbq+faBG6t+pOfGqh/pWV81dmQ1TTorvB6aVLSO2TqR/5qhtA139L8YEA9wiDxvHMwEWDoRqcATNGz7c0D0ej60XX+rHKcJOB7/YQNNcIWLQM+NPsViF3XPuVi0pT/p3bgUcayKgtXNDHg7Lk1bnEdLhF6UyKCrj5TSoeQfQ+5ndVOZc4GOy6pzVKI2PC+WF9zYLg9C4/TjyQ8/vn1rf398/urfGpVJuBeqSKudhvt6KrU6GTYOMAZY34+/XCu2JWTsNPKJ5yCiysFi+giAT5gLuXVwfXT+JYpYYeIOAQB9bZCiKQ5oc+odRHmXoryppB0C5lcZApSb8tQNWuhkq1wDuKwogJDLO5Qwh67DsG4E52IqMpXuCt17gxqqAcapWA1SpE20pOm6CiLw31ykoJYb2dMma/cZclYL59WbFTq38YsE6L+5yk/el1HtYL/X7+nFg23HoK2krcyfVi9pNKwRXWhBkE1ItuXcq/aBameb7RgOIOJiWL5hqPIO/U/nJQyHxvpbxepVT6RCRQIp+lZkbHlH1XiF1kohnEs9VWnbq798UtV5YTxrp3nIxQulrhtcMv4gsc+C365Sk/2BLFvxZZl4d4slKgIs6CW7b+Hwk8smBD+cj80+YvgRyG4waJVd47rkbXKJAThBssrpqtdvy+AEoz9BxYoIYOnyC7+TonWsnc3FYpkNKGvMjsDcbFqMQpRE4Yj5X42Vb9+azIhZl/VRmb0g6hw2B11baSQofWY0G4sDY7Y34WRbmiidLXNNRNzyVlm6rS+VgNm38ZZ2WDm+5kT/hqS1A+6Ar0RLay3Nr1U/3thrpX/gaujy1lqL4pNctqmtC8AsCrTxMh3FCaibLwzElabM4ZibKv9bBNQ6UBdSa4dKUFubpbC2Nm8R2Pb+ldC29tEFt7VTKbzWHq0s/+cL8aGN800n3boSDaV9Jvl/YftHKhaUjw19SdZ6m5QraKbR0UfCH7vgKb3zpzcGsdbWtsWdx75i3oSnXemGVW4xSx3M25h6AumcKjGFhqAordAghu8BC7ZLISV0n0z+VdZtEkbYle9EyVaEXoLylXJagGDS8o4mYHDxv5iNZn388Byv2dxk/fhjuI6wQ8brtFsI12V2gFCNJS/K7mK3NNxrTG7IvmaDGuJfwlcFizA7hqMkRlpzLIGLv+Y8x/RQkXRAlniYJW3Qky/llKmMCC9fKGQWdN1GtbJrJ9i8e3/SlzWMIO5uzxD4L9sAbNPryHsKA7R135wkEjSLpfI8EN0Yks6WsZp+i88iw8iH6OjP9DfRrV4HfB3McthO76Gu4NI3HrG4/YX1AuTo+RxYAYVe/yQc02+pZDgri5SKdKeWAIsljsZ4R6iEViT/05uHGbQGsQLP0luVFjshHF+D5KhVRRarzHIFAjO8DWqytF2ePZXWFTyVJUFYdYX/L0TlgMosGZxRtDcb/w2iztKaRvRa2PK43u3/Tp3b/yfqW4uuNVRQrdhyQ+6kRpuSFnh8E8bvdOt9Nc1ts2JNH9T02sUL31WmhbbaSDBLqzobmnMwquPe6LmhiMy0V97JbHYp8uirlrYyPG9x8br2mI8GPDpHKDnV8u8RbCxHBRy2wfOqCe9ID1H9qQBdYM1qUSqPReQHmX19A78cV0+tbiPm+9tpXeYHmH08oMTj48TkquTyYtBBXAP9cdzDnf8FatneJOxUAAA="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave120")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave120-in-process-stability-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave120.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave120.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_TELEMETRY": "1"}

    phase = "event-profile"
    measured = run([exe, MODEL, "profile"], cwd=TREE, env=prod_env, check=False)
    save("event-profile.log", measured)
    if measured.returncode or "[wave120-profile]" not in measured.stdout:
        raise RuntimeError("single-pass production profile failed")

    profile = json.loads(re.search(r"\[wave120-profile\]\s*(\{[^\n]+\})", measured.stdout).group(1))
    stages = [json.loads(x) for x in re.findall(r"\[wave120-stage\]\s*(\{[^\n]+\})", measured.stdout)]
    if len(stages) != 9 or profile["gpu_prefill_ms"] <= 0:
        raise RuntimeError(f"event profile contract failed: {profile}, {len(stages)} stages")
    stage_sum = sum(x["total_ms"] for x in stages)
    summary = {"wave": 120, "gpu": fields, "model": model_meta,
               "profile": profile, "stages": stages, "stage_sum_ms": stage_sum,
               "stage_sum_over_gpu_total": stage_sum / profile["gpu_prefill_ms"],
               "retention_authority": False,
               "target_15000_tps_achieved": profile["gpu_prefill_tps"] >= 15000}
    (RESULTS / "wave120-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE120_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
